# E-commerce Customer Analysis (SQL)

This project uses SQL (DuckDB) to analyse customer behaviour, including spending patterns, segmentation, and churn risk, to support data-driven business decisions.

In [2]:
!pip install duckdb


[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import pandas as pd
import duckdb

df = pd.read_csv("E-commerce Customer Behavior - Sheet1.csv")
df.head()

,Customer ID,Gender,Age,City,Membership Type,Total Spend,Items Purchased,Average Rating,Discount Applied,Days Since Last Purchase,Satisfaction Level
0,101,Female,29,New York,Gold,1120.20,14,4.6,True,25,Satisfied
1,102,Male,34,Los Angeles,Silver,780.50,11,4.1,False,18,Neutral
2,103,Female,43,Chicago,Bronze,510.75,9,3.4,True,42,Unsatisfied
3,104,Male,30,San Francisco,Gold,1480.30,19,4.7,False,12,Satisfied
4,105,Male,27,Miami,Silver,720.40,13,4.0,True,55,Unsatisfied


In [4]:
duckdb.register("customers", df)

In [5]:
# Overview of key customer metrics including total count, average spending, purchasing behavior, and recency
query = """
SELECT
    COUNT(*) AS total_customers,
    ROUND(AVG("Total Spend"), 2) AS avg_spend,
    ROUND(AVG("Items Purchased"), 2) AS avg_items,
    ROUND(AVG("Days Since Last Purchase"), 2) AS avg_recency
FROM customers;
"""

duckdb.query(query).to_df()

,total_customers,avg_spend,avg_items,avg_recency
0,350,845.38,12.6,26.59


In [6]:
# Analyze customer distribution and revenue contribution across different membership tiers
query = """
SELECT
    "Membership Type",
    COUNT(*) AS customer_count,
    ROUND(SUM("Total Spend"), 2) AS total_revenue,
    ROUND(AVG("Total Spend"), 2) AS avg_spend,
    ROUND(AVG("Items Purchased"), 2) AS avg_items
FROM customers
GROUP BY "Membership Type"
ORDER BY total_revenue DESC;
"""

duckdb.query(query).to_df()

,Membership Type,customer_count,total_revenue,avg_spend,avg_items
0,Gold,117,153403.9,1311.14,17.62
1,Silver,117,87566.6,748.43,11.66
2,Bronze,116,54913.1,473.39,8.49


In [7]:
# Segment customers into high, medium, and low value groups based on total spending
query = """
SELECT
    "Customer ID",
    "Total Spend",
    CASE
        WHEN "Total Spend" >= 800 THEN 'High Value'
        WHEN "Total Spend" >= 400 THEN 'Medium Value'
        ELSE 'Low Value'
    END AS value_segment
FROM customers
ORDER BY "Total Spend" DESC;
"""

duckdb.query(query).to_df()

,Customer ID,Total Spend,value_segment
0,110,1520.1,High Value
1,128,1500.1,High Value
2,218,1500.1,High Value
3,290,1500.1,High Value
4,331,1500.1,High Value
...,...,...,...
345,232,420.8,Medium Value
346,202,420.8,Medium Value
347,142,420.8,Medium Value
348,315,420.8,Medium Value


In [8]:
# Summarize customer count and average spending for each value segment
query = """
SELECT
    CASE
        WHEN "Total Spend" >= 800 THEN 'High Value'
        WHEN "Total Spend" >= 400 THEN 'Medium Value'
        ELSE 'Low Value'
    END AS value_segment,
    COUNT(*) AS customer_count,
    ROUND(AVG("Total Spend"), 2) AS avg_spend
FROM customers
GROUP BY value_segment
ORDER BY avg_spend DESC;
"""

duckdb.query(query).to_df()

,value_segment,customer_count,avg_spend
0,High Value,158,1182.30
1,Medium Value,192,568.13


In [9]:
# Calculate churn risk rate based on customers inactive for more than 30 days
query = """
SELECT
    COUNT(*) AS total_customers,
    SUM(CASE WHEN "Days Since Last Purchase" > 30 THEN 1 ELSE 0 END) AS churn_risk,
    ROUND(
        SUM(CASE WHEN "Days Since Last Purchase" > 30 THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS churn_rate
FROM customers;
"""

duckdb.query(query).to_df()

,total_customers,churn_risk,churn_rate
0,350,124.0,35.43


In [10]:
# Identify high-value customers who are at risk of churn
query = """
SELECT
    "Customer ID",
    "Total Spend",
    "Days Since Last Purchase",
    "Membership Type"
FROM customers
WHERE "Total Spend" >= 800
  AND "Days Since Last Purchase" > 30
ORDER BY "Total Spend" DESC;
"""

duckdb.query(query).to_df()

,Customer ID,Total Spend,Days Since Last Purchase,Membership Type
0,382,1160.6,31,Gold
1,406,1160.6,33,Gold
2,430,1160.6,35,Gold
3,394,1140.6,32,Gold
4,418,1140.6,34,Gold
5,442,1140.6,36,Gold


In [11]:
# Rank customers by total spending using window functions
query = """
SELECT
    "Customer ID",
    "Total Spend",
    RANK() OVER (ORDER BY "Total Spend" DESC) AS rank
FROM customers
LIMIT 10;
"""

duckdb.query(query).to_df()

,Customer ID,Total Spend,rank
0,110,1520.1,1
1,128,1500.1,2
2,218,1500.1,2
3,290,1500.1,2
4,331,1500.1,2
5,188,1500.1,2
6,260,1500.1,2
7,158,1500.1,2
8,433,1490.1,9
9,409,1490.1,9


## Key Insights

- Customer spending varies significantly across membership tiers
- High-value customers contribute the majority of total revenue
- A portion of high-value customers shows churn risk due to inactivity
- Customer segmentation based on spending helps identify target groups

## Business Implications

- Prioritise retention strategies for high-value customers at risk
- Use personalised promotions to improve engagement
- Leverage segmentation to optimise marketing campaigns
- Monitor recency to detect potential churn early